# MOM6-Solo Geometry Evolution

Visualise a generated MOM6-solo wettable-envelope run against its prescribed geometry file. The notebook compares the initial geometry, the latest archived model state, and the 100-year prescribed end-state reference.

The model restart contains shelf thickness, shelf area, and shelf mass. It does not contain ice-base elevation directly, so the 3-D model-time surface uses the prescribed geometry interpolated to the latest restart time, with restart fields used for consistency checks and time-series diagnostics.

In [ ]:
from pathlib import Path

# Edit these if your 10-year continuation lives under a different generated case name.
CASE_NAME = "ocean3-generated-geometry-envelope-year"
CONTROL = Path(f"/scratch/au88/jr5971/mom6-isomip-generated-controls/{CASE_NAME}")
ARCHIVE = Path(f"/scratch/au88/jr5971/mom6-isomip-generated/{CASE_NAME}/archive/{CASE_NAME}")

# Geometry snapshots to compare in 3-D. Index 100 is the prescribed 100-year end-state.
REFERENCE_INDEX = 100
VERTICAL_EXAGGERATION = 40.0
PLOT_STRIDE = 1

print("control:", CONTROL)
print("archive:", ARCHIVE)

In [ ]:
import re
import warnings

import netCDF4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import plotly.graph_objects as go
except ImportError:
    go = None
    warnings.warn("plotly is not available; 3-D interactive plots will be skipped")

RHO_ICE = 900.0
SECONDS_PER_NOLEAP_YEAR = 365.0 * 86400.0
DAYS_PER_NOLEAP_YEAR = 365.0
FILL_LARGE = 1.0e30

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

In [ ]:
def numbered_dirs(root: Path, prefix: str) -> list[Path]:
    pat = re.compile(rf"^{re.escape(prefix)}(\d+)$")
    dirs = []
    if not root.exists():
        return dirs
    for path in root.iterdir():
        if path.is_dir() and pat.match(path.name):
            dirs.append(path)
    return sorted(dirs, key=lambda p: int(p.name[len(prefix):]))


def latest_numbered_dir(root: Path, prefix: str) -> Path | None:
    dirs = numbered_dirs(root, prefix)
    return dirs[-1] if dirs else None


def find_one(patterns: list[str], roots: list[Path]) -> Path:
    matches = []
    for root in roots:
        for pattern in patterns:
            matches.extend(sorted(root.glob(pattern)))
    if not matches:
        raise FileNotFoundError(f"Could not find any of {patterns} under {roots}")
    return matches[0]


def nc_array(ds: netCDF4.Dataset, name: str) -> np.ndarray:
    var = ds.variables[name]
    data = np.asarray(var[:], dtype="f8")
    fill = getattr(var, "_FillValue", None)
    missing = getattr(var, "missing_value", None)
    data[np.abs(data) > FILL_LARGE] = np.nan
    if fill is not None:
        data[data == fill] = np.nan
    if missing is not None:
        data[data == missing] = np.nan
    return data


def axis_cell_area(x_m: np.ndarray, y_m: np.ndarray) -> float:
    dx = float(np.nanmedian(np.diff(x_m))) if x_m.size > 1 else 1.0
    dy = float(np.nanmedian(np.diff(y_m))) if y_m.size > 1 else 1.0
    return abs(dx * dy)


def interp_time_series(time_years: np.ndarray, field: np.ndarray, target_year: float) -> np.ndarray:
    if target_year <= time_years[0]:
        return field[0].copy()
    if target_year >= time_years[-1]:
        return field[-1].copy()
    hi = int(np.searchsorted(time_years, target_year))
    lo = hi - 1
    w = (target_year - time_years[lo]) / (time_years[hi] - time_years[lo])
    return (1.0 - w) * field[lo] + w * field[hi]


def geometry_metrics(label: str, year: float, thick: np.ndarray, frac: np.ndarray, cell_area_m2: float, mass: np.ndarray | None = None) -> dict[str, float | str]:
    active = np.isfinite(thick) & np.isfinite(frac) & (frac > 1.0e-10) & (thick > 1.0e-6)
    area = np.where(active, frac * cell_area_m2, 0.0)
    volume = np.where(active, area * thick, 0.0)
    if mass is None:
        mass_total = float(np.nansum(RHO_ICE * thick * np.where(active, 1.0, 0.0) * area))
    else:
        mass_total = float(np.nansum(np.where(np.isfinite(mass), mass, 0.0) * area))
    return {
        "label": label,
        "year": float(year),
        "floating_cells": int(np.count_nonzero(active)),
        "shelf_area_km2": float(np.nansum(area) / 1.0e6),
        "ice_volume_km3": float(np.nansum(volume) / 1.0e9),
        "shelf_mass_gt": mass_total / 1.0e12,
        "mean_h_m": float(np.nanmean(np.where(active, thick, np.nan))),
        "max_h_m": float(np.nanmax(np.where(active, thick, np.nan))),
    }


def category_table(cat: np.ndarray, years: np.ndarray) -> pd.DataFrame:
    rows = []
    for n, year in enumerate(years):
        vals = cat[n]
        vals = vals[np.isfinite(vals)]
        codes, counts = np.unique(vals.astype(int), return_counts=True)
        row = {"year": float(year)}
        row.update({f"cat_{int(code)}": int(count) for code, count in zip(codes, counts)})
        rows.append(row)
    return pd.DataFrame(rows).fillna(0)

In [ ]:
geometry_file = find_one(["INPUT/generated_geometry/*_geometry.nc"], [CONTROL])
topography_file = find_one(["INPUT/generated_geometry/*_topog.nc"], [CONTROL])
latest_restart = latest_numbered_dir(ARCHIVE, "restart")
latest_output = latest_numbered_dir(ARCHIVE, "output")

if latest_restart is None:
    raise FileNotFoundError(f"No restartNNN directories found under {ARCHIVE}")

print("geometry file:", geometry_file)
print("topography file:", topography_file)
print("latest restart:", latest_restart)
print("latest output:", latest_output)

In [ ]:
with netCDF4.Dataset(geometry_file) as ds:
    x = nc_array(ds, "x")
    y = nc_array(ds, "y")
    time_s = nc_array(ds, "Time")
    geom_years = time_s / SECONDS_PER_NOLEAP_YEAR
    thick = nc_array(ds, "thick")
    base = nc_array(ds, "base_elevation")
    frac = nc_array(ds, "floating_fraction")
    bed = nc_array(ds, "bed_elevation")
    envelope = nc_array(ds, "wettable_envelope") if "wettable_envelope" in ds.variables else None

with netCDF4.Dataset(topography_file) as ds:
    topog_depth = nc_array(ds, "depth")
    topog_envelope = nc_array(ds, "wettable_envelope") if "wettable_envelope" in ds.variables else None
    generated_wet = nc_array(ds, "generated_wet_mask") if "generated_wet_mask" in ds.variables else None

cell_area_m2 = axis_cell_area(x, y)
print(f"geometry snapshots: {len(geom_years)} from year {geom_years[0]:.1f} to {geom_years[-1]:.1f}")
print(f"grid: nx={x.size}, ny={y.size}, cell_area={cell_area_m2:.3e} m2")

In [ ]:
restart_file = latest_restart / "Shelf.res.nc"
with netCDF4.Dataset(restart_file) as ds:
    restart_days = float(nc_array(ds, "Time")[-1])
    restart_year = restart_days / DAYS_PER_NOLEAP_YEAR
    restart_h = nc_array(ds, "h_shelf")[-1]
    restart_area = nc_array(ds, "shelf_area")[-1]
    restart_mass = nc_array(ds, "shelf_mass")[-1]

restart_frac = np.divide(restart_area, cell_area_m2, out=np.zeros_like(restart_area), where=cell_area_m2 > 0.0)
prescribed_h_now = interp_time_series(geom_years, thick, restart_year)
prescribed_base_now = interp_time_series(geom_years, base, restart_year)
prescribed_frac_now = interp_time_series(geom_years, frac, restart_year)
prescribed_bed_now = interp_time_series(geom_years, bed, restart_year)

ref_index = min(max(int(REFERENCE_INDEX), 0), len(geom_years) - 1)

print("latest restart file:", restart_file)
print(f"latest restart time: {restart_days:.3f} days = year {restart_year:.3f}")
print(f"reference geometry index: {ref_index}, year {geom_years[ref_index]:.1f}")
print("max |restart shelf_mass - 900*h_shelf|:", float(np.nanmax(np.abs(restart_mass - RHO_ICE * restart_h))))
print("max |restart h - prescribed h at restart time|:", float(np.nanmax(np.abs(restart_h - prescribed_h_now))))
print("max |restart area fraction - prescribed floating_fraction|:", float(np.nanmax(np.abs(restart_frac - prescribed_frac_now))))

In [ ]:
summary = pd.DataFrame([
    geometry_metrics("prescribed index 0", geom_years[0], thick[0], frac[0], cell_area_m2),
    geometry_metrics("model restart", restart_year, restart_h, restart_frac, cell_area_m2, mass=restart_mass),
    geometry_metrics("prescribed at restart time", restart_year, prescribed_h_now, prescribed_frac_now, cell_area_m2),
    geometry_metrics(f"prescribed index {ref_index}", geom_years[ref_index], thick[ref_index], frac[ref_index], cell_area_m2),
])
summary

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
extent = [x.min() / 1000, x.max() / 1000, y.min() / 1000, y.max() / 1000]

panels = [
    (thick[0], "thickness, index 0", "m"),
    (prescribed_h_now, f"thickness, restart year {restart_year:.2f}", "m"),
    (thick[ref_index], f"thickness, index {ref_index}", "m"),
    (frac[0], "floating fraction, index 0", "1"),
    (prescribed_frac_now, f"floating fraction, restart year {restart_year:.2f}", "1"),
    (frac[ref_index], f"floating fraction, index {ref_index}", "1"),
]

for ax, (field, title, units) in zip(axes.ravel(), panels):
    im = ax.imshow(field, origin="lower", extent=extent, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(units)

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)

change_panels = [
    (prescribed_h_now - thick[0], "thickness change: restart - index 0", "m", "RdBu_r"),
    (thick[ref_index] - thick[0], f"thickness change: index {ref_index} - index 0", "m", "RdBu_r"),
    (restart_h - prescribed_h_now, "restart h - prescribed h", "m", "RdBu_r"),
    (prescribed_frac_now - frac[0], "fraction change: restart - index 0", "1", "RdBu_r"),
    (frac[ref_index] - frac[0], f"fraction change: index {ref_index} - index 0", "1", "RdBu_r"),
    (restart_frac - prescribed_frac_now, "restart fraction - prescribed fraction", "1", "RdBu_r"),
]

for ax, (field, title, units, cmap) in zip(axes.ravel(), change_panels):
    vmax = np.nanpercentile(np.abs(field), 99) if np.isfinite(field).any() else 1.0
    vmax = max(float(vmax), 1.0e-12)
    im = ax.imshow(field, origin="lower", extent=extent, aspect="auto", cmap=cmap, vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(units)

plt.show()

In [ ]:
output_dirs = numbered_dirs(ARCHIVE, "output")
rows = []
category_frames = []

for out in output_dirs:
    ice = out / "ice.nc"
    if not ice.exists():
        continue
    with netCDF4.Dataset(ice) as ds:
        if "Time" not in ds.variables or "h_shelf" not in ds.variables:
            continue
        time_days = nc_array(ds, "Time")
        years = time_days / DAYS_PER_NOLEAP_YEAR
        h = nc_array(ds, "h_shelf")
        area_h = nc_array(ds, "area_shelf_h")
        mass = nc_array(ds, "shelf_mass")
        cat = nc_array(ds, "dynamic_cavity_category") if "dynamic_cavity_category" in ds.variables else None
        for n, year in enumerate(years):
            valid_h = np.isfinite(h[n]) & (h[n] > 0.0)
            rows.append({
                "output": out.name,
                "year": float(year),
                "shelf_area_km2": float(np.nansum(area_h[n]) / 1.0e6),
                "shelf_mass_gt": float(np.nansum(mass[n] * area_h[n]) / 1.0e12),
                "mean_h_m": float(np.nanmean(np.where(valid_h, h[n], np.nan))),
                "floating_cells": int(np.count_nonzero(valid_h)),
                "max_mass_minus_900h": float(np.nanmax(np.abs(mass[n] - RHO_ICE * h[n]))),
            })
        if cat is not None and np.isfinite(cat).any():
            frame = category_table(cat, years)
            frame.insert(0, "output", out.name)
            category_frames.append(frame)

ts = pd.DataFrame(rows).sort_values("year").drop_duplicates(subset=["year"], keep="last")
categories = pd.concat(category_frames, ignore_index=True).sort_values("year") if category_frames else pd.DataFrame()

print(f"loaded {len(ts)} diagnostic time samples from {len(output_dirs)} output directories")
ts.tail()

In [ ]:
if ts.empty:
    print("No non-fill ice.nc time series found. Use restart-based plots above instead.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    axes[0, 0].plot(ts["year"], ts["shelf_area_km2"], marker="o")
    axes[0, 0].set_ylabel("shelf area (km$^2$)")
    axes[0, 0].set_xlabel("model year")

    axes[0, 1].plot(ts["year"], ts["shelf_mass_gt"], marker="o")
    axes[0, 1].set_ylabel("shelf mass (Gt)")
    axes[0, 1].set_xlabel("model year")

    axes[1, 0].plot(ts["year"], ts["mean_h_m"], marker="o")
    axes[1, 0].set_ylabel("mean shelf thickness (m)")
    axes[1, 0].set_xlabel("model year")

    axes[1, 1].plot(ts["year"], ts["floating_cells"], marker="o")
    axes[1, 1].set_ylabel("cells with h_shelf > 0")
    axes[1, 1].set_xlabel("model year")

    plt.show()

In [ ]:
if categories.empty:
    print("No dynamic_cavity_category time series found, or diagnostics were all fill values.")
else:
    cat_labels = {
        "cat_0": "0 background",
        "cat_1": "1 open cavity",
        "cat_2": "2 floating shelf",
        "cat_3": "3 grounded/closed",
        "cat_4": "4 thin cavity",
        "cat_5": "5 outside envelope",
        "cat_6": "6 would dry",
    }
    cat_cols = [c for c in categories.columns if c.startswith("cat_")]
    fig, ax = plt.subplots(figsize=(12, 5))
    for col in cat_cols:
        if categories[col].max() > 0:
            ax.plot(categories["year"], categories[col], marker="o", label=cat_labels.get(col, col))
    ax.set_xlabel("model year")
    ax.set_ylabel("cell count")
    ax.legend(loc="best")
    plt.show()

    display(categories.tail())

In [ ]:
def plot_3d_geometry():
    if go is None:
        print("plotly is not available")
        return None

    stride = max(int(PLOT_STRIDE), 1)
    xx, yy = np.meshgrid(x[::stride] / 1000.0, y[::stride] / 1000.0)
    bed0 = bed[0, ::stride, ::stride]
    base0 = base[0, ::stride, ::stride]
    surf0 = base0 + thick[0, ::stride, ::stride]
    basen = prescribed_base_now[::stride, ::stride]
    surfn = basen + prescribed_h_now[::stride, ::stride]
    baseref = base[ref_index, ::stride, ::stride]
    surfref = baseref + thick[ref_index, ::stride, ::stride]

    fig = go.Figure()
    fig.add_surface(x=xx, y=yy, z=bed0 * VERTICAL_EXAGGERATION, name="bed", colorscale="Greys", opacity=0.55, showscale=False)
    fig.add_surface(x=xx, y=yy, z=base0 * VERTICAL_EXAGGERATION, name="base index 0", colorscale="Blues", opacity=0.45, showscale=False)
    fig.add_surface(x=xx, y=yy, z=surf0 * VERTICAL_EXAGGERATION, name="surface index 0", colorscale="Blues", opacity=0.25, showscale=False)
    fig.add_surface(x=xx, y=yy, z=basen * VERTICAL_EXAGGERATION, name="base latest", colorscale="Viridis", opacity=0.65, showscale=False)
    fig.add_surface(x=xx, y=yy, z=surfn * VERTICAL_EXAGGERATION, name="surface latest", colorscale="Viridis", opacity=0.35, showscale=False)
    fig.add_surface(x=xx, y=yy, z=baseref * VERTICAL_EXAGGERATION, name=f"base index {ref_index}", colorscale="Reds", opacity=0.35, showscale=False)
    fig.add_surface(x=xx, y=yy, z=surfref * VERTICAL_EXAGGERATION, name=f"surface index {ref_index}", colorscale="Reds", opacity=0.22, showscale=False)

    dx = (x.max() - x.min()) / 1000.0
    dy = (y.max() - y.min()) / 1000.0
    dz = np.nanmax([surf0.max(), surfn.max(), surfref.max()]) - np.nanmin(bed0)
    fig.update_layout(
        title=f"Ocean3 geometry: index 0 vs restart year {restart_year:.2f} vs index {ref_index}",
        scene=dict(
            xaxis_title="x (km)",
            yaxis_title="y (km)",
            zaxis_title=f"z x {VERTICAL_EXAGGERATION:g}",
            aspectmode="manual",
            aspectratio=dict(x=1.0, y=max(dy / dx, 1.0e-6), z=max((dz * VERTICAL_EXAGGERATION / 1000.0) / dx, 0.05)),
        ),
        legend=dict(orientation="h"),
        height=750,
    )
    return fig


fig = plot_3d_geometry()
if fig is not None:
    fig.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)

env = topog_envelope if topog_envelope is not None else (np.nanmax(frac, axis=0) > 0).astype(float)
future = ((env > 0.5) & ~(frac[0] > 1.0e-10)).astype(float)
active_now = (prescribed_frac_now > 1.0e-10).astype(float)

for ax, field, title in zip(
    axes,
    [env, future, active_now],
    ["wettable envelope", "future expansion cells", f"active shelf at year {restart_year:.2f}"],
):
    im = ax.imshow(field, origin="lower", extent=extent, aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    fig.colorbar(im, ax=ax)

plt.show()

print("envelope cells:", int(np.count_nonzero(env > 0.5)))
print("future-expansion cells relative to index 0:", int(np.count_nonzero(future > 0.5)))
print("active shelf cells at latest restart:", int(np.count_nonzero(active_now > 0.5)))

## Interpretation Checklist

- The latest restart year should match the number of completed noleap years in the payu archive.
- `restart h - prescribed h` and `restart fraction - prescribed fraction` should be close to zero except for tiny interpolation/threshold differences.
- Category `5 outside envelope` and category `6 would dry` should remain zero. If either appears, the wettable-envelope assumption has been violated.
- A changing `shelf_area_km2`, `shelf_mass_gt`, `mean_h_m`, or category count demonstrates that MOM is not using a static shelf state.
- The 3-D figure uses the prescribed model-time base/surface because MOM restarts do not store base elevation directly; the restart fields are checked separately against the same prescribed geometry.